# Tiny Recursive Reasoning Model (TRM) Workspace

This notebook is optimized for training and evaluating TRM models from scratch in the Modal environment.

In [2]:
# ── 1. Environment and Path Setup ───────────────────────────────────────────
import os
import sys
from pathlib import Path

os.chdir("/root/EdgeTRM")
print("Working Directory:", os.getcwd())
# !git fetch origin
# !git reset --hard origin/main
!git pull origin main
# Add TinyRecursiveModels to system path
repo_root = Path.cwd()
trm_root = repo_root / "TinyRecursiveModels"
if str(trm_root) not in sys.path:
    sys.path.insert(0, str(trm_root))
print("trm_root added to sys.path:", trm_root)

Working Directory: /__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f
From https://github.com/Seqaeon/EdgeTRM
 * branch            main       -> FETCH_HEAD
Already up to date.
trm_root added to sys.path: /__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f/TinyRecursiveModels


In [1]:
# ── 2. Fix Duplicate Modules on Modal ────────────────────────────────────────
# Replaces duplicate trm.py with a symlink to prevent dual-import namespace conflicts
modal_top = "/__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f/TinyRecursiveModels/trm.py"
modal_sub = "/__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f/TinyRecursiveModels/models/recursive_reasoning/trm.py"

if os.path.exists(modal_sub) and os.path.exists(modal_top) and not os.path.islink(modal_top):
    os.rename(modal_top, modal_top + ".bak")
    os.symlink(modal_sub, modal_top)
    print("✓ Successfully symlinked trm.py files to resolve namespace conflicts!")
else:
    print("✓ Symlink already exists or paths are aligned.")

NameError: name 'os' is not defined

In [3]:
!uv pip install --system {trm_root}
%uv pip install einops

Using Python 3.12.6 environment at: /usr/local
Resolved 64 packages in 2.58s
Building antlr4-python3-runtime==4.9.3
Building antlr4-python3-runtime==4.9.3
Building tiny-recursive-models @ file:///__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f/T
Building antlr4-python3-runtime==4.9.3
Building tiny-recursive-models @ file:///__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f/T
Building adam-atan2==0.0.3
Building antlr4-python3-runtime==4.9.3
Building tiny-recursive-models @ file:///__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f/T
Building adam-atan2==0.0.3
⠙ Preparing packages... (0/35)
Building antlr4-python3-runtime==4.9.3
Building tiny-recursive-models @ file:///__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f/T
Building adam-atan2==0.0.3
⠙ Preparing packages... (0/35)
Building antlr4-python3-runtime==4.9.3
Building tiny-recursive-models @ file:///__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f/T
Building adam-atan2==0.0.3
⠙ Preparing packages... (0/35)
smmap      ------------------------------     0 B/23.82 KiB
Bui

In [4]:
# ── 3. High-Performance Per-Puzzle Evaluator ─────────────────────────────────
import os, json, time, math, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.notebook import tqdm

def evaluate_arc_per_puzzle(model, loader, device="cuda", n_sup_max=16, max_batches=None, return_pass2=False, fast_mode=True):
    """
    Unified high-performance evaluator.
    If the dataset is Maze-Hard/Sudoku-Extreme (num_puzzle_identifiers == 1),
    it runs simple exact-match and cell accuracy (no test-time ensembling).
    If it is ARC-AGI, it runs full ensembled voting.
    """
    model.eval()
    inner = get_inner(model)
    if hasattr(inner, "model"):
        inner = inner.model
    
    # Check if we are running Maze/Sudoku (where max identifier <= 1)
    ds = loader.dataset
    is_maze_or_sudoku = False
    if hasattr(ds, 'num_puzzle_identifiers') and ds.num_puzzle_identifiers <= 1:
        is_maze_or_sudoku = True
    elif hasattr(ds, 'metadata') and ds.metadata.num_puzzle_identifiers <= 1:
        is_maze_or_sudoku = True
        
    if is_maze_or_sudoku:
        # Standard fast exact-match evaluation for Maze/Sudoku
        correct_puzzles = 0
        total_puzzles = 0
        correct_cells = 0
        total_cells = 0
        t0 = time.time()
        
        # Determine batch size dynamically
        batch_size = loader.batch_size if (hasattr(loader, "batch_size") and loader.batch_size is not None) else 512
        
        # Bypass DataLoader collation completely if needed for high performance
        try:
            ds._lazy_load_dataset()
            set_name = list(ds._data.keys())[0]
            dataset = ds._data[set_name]
            inputs_np = dataset["inputs"]
            labels_np = dataset["labels"]
            
            num_samples = len(inputs_np)
            num_batches = (num_samples + batch_size - 1) // batch_size
            
            with torch.no_grad():
                pbar = tqdm(range(num_batches), desc="Evaluating Maze Batches", leave=False)
                for batch_idx in pbar:
                    if max_batches is not None and batch_idx >= max_batches:
                        break
                    
                    start_idx = batch_idx * batch_size
                    end_idx = min(start_idx + batch_size, num_samples)
                    
                    x_batch = torch.from_numpy(inputs_np[start_idx:end_idx].copy()).to(device, dtype=torch.long)
                    y_true  = torch.from_numpy(labels_np[start_idx:end_idx].copy()).to(device, dtype=torch.long)
                    pids    = torch.zeros(end_idx - start_idx, dtype=torch.long, device=device) # Single default pid
                    
                    batch = {
                        "inputs":             x_batch.to(torch.int32),
                        "labels":             y_true.to(torch.int32),
                        "puzzle_identifiers": pids.to(torch.int32),
                    }
                    
                    carry = inner.initial_carry(batch)
                    cast  = lambda t: t.to(device)
                    ic    = carry.inner_carry
                    
                    from models.recursive_reasoning.trm import TinyRecursiveReasoningModel_ACTV1Carry, TinyRecursiveReasoningModel_ACTV1InnerCarry
                    carry = TinyRecursiveReasoningModel_ACTV1Carry(
                        inner_carry=TinyRecursiveReasoningModel_ACTV1InnerCarry(
                            z_H=cast(ic.z_H), z_L=cast(ic.z_L)),
                        steps=carry.steps.to(device),
                        halted=carry.halted.to(device),
                        current_data={k: v.to(device) for k, v in carry.current_data.items()},
                    )
                    
                    for _ in range(n_sup_max):
                        carry, outputs = inner(carry, batch)
                        if carry.halted.all():
                            break
                            
                    preds = outputs["logits"].argmax(-1) # (B, seq_len)
                    
                    mask = (y_true >= 0)
                    for i in range(preds.size(0)):
                        p_i = preds[i][mask[i]]
                        y_i = y_true[i][mask[i]]
                        
                        is_exact = torch.equal(p_i, y_i)
                        if is_exact:
                            correct_puzzles += 1
                        total_puzzles += 1
                        
                        correct_cells += (p_i == y_i).sum().item()
                        total_cells += y_i.numel()
        except Exception as e:
            # Fallback to standard dataloader iteration
            with torch.no_grad():
                for batch_idx, batch in enumerate(loader):
                    if max_batches is not None and batch_idx >= max_batches:
                        break
                    
                    # Unpack PuzzleDataset or ARCDataset batches correctly
                    if isinstance(batch, (list, tuple)) and len(batch) == 3 and isinstance(batch[1], dict):
                        batch_dict = batch[1]
                    elif isinstance(batch, dict):
                        batch_dict = batch
                    elif isinstance(batch, (list, tuple)):
                        x_batch, y_true, pids = batch
                        batch_dict = {
                            "inputs": x_batch,
                            "labels": y_true,
                            "puzzle_identifiers": pids
                        }
                    else:
                        batch_dict = batch
                    
                    # Move to device
                    batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch_dict.items()}
                    y_true = batch["labels"]
                    
                    carry = inner.initial_carry(batch)
                    cast  = lambda t: t.to(device)
                    ic    = carry.inner_carry
                    
                    from models.recursive_reasoning.trm import TinyRecursiveReasoningModel_ACTV1Carry, TinyRecursiveReasoningModel_ACTV1InnerCarry
                    carry = TinyRecursiveReasoningModel_ACTV1Carry(
                        inner_carry=TinyRecursiveReasoningModel_ACTV1InnerCarry(
                            z_H=cast(ic.z_H), z_L=cast(ic.z_L)),
                        steps=carry.steps.to(device),
                        halted=carry.halted.to(device),
                        current_data={k: v.to(device) for k, v in carry.current_data.items()},
                    )
                    
                    for _ in range(n_sup_max):
                        carry, outputs = inner(carry, batch)
                        if carry.halted.all():
                            break
                            
                    preds = outputs["logits"].argmax(-1)
                    
                    mask = (y_true >= 0)
                    for i in range(preds.size(0)):
                        p_i = preds[i][mask[i]]
                        y_i = y_true[i][mask[i]]
                        
                        is_exact = torch.equal(p_i, y_i)
                        if is_exact:
                            correct_puzzles += 1
                        total_puzzles += 1
                        
                        correct_cells += (p_i == y_i).sum().item()
                        total_cells += y_i.numel()
                        
        elapsed = time.time() - t0
        ms_per_puzzle = (elapsed / total_puzzles) * 1000 if total_puzzles > 0 else 0.0
        exact_acc = correct_puzzles / total_puzzles if total_puzzles > 0 else 0.0
        cell_acc = correct_cells / total_cells if total_cells > 0 else 0.0
        
        if return_pass2:
            return exact_acc, exact_acc, cell_acc, ms_per_puzzle, total_puzzles
        else:
            return exact_acc, cell_acc, ms_per_puzzle, total_puzzles
            
    # Legacy ARC-AGI ensembled evaluator logic
    # Pre-harvest all test puzzles from disk to prevent repeated slow JSON parsing
    test_puzzles = {}
    test_dir = os.path.join(os.path.dirname(os.path.dirname(loader.dataset.inputs.base.filename if hasattr(loader.dataset.inputs, 'base') else ".")), "test") if hasattr(loader.dataset, 'inputs') else "data1/arc2test-aug-128/test"
    if not os.path.exists(test_dir):
        test_dir = "data1/arc2test-aug-128/test"
    
    # Locate challenge files
    challenge_path = "data1/arc-agi_test_challenges.json"
    if not os.path.exists(challenge_path):
        challenge_path = "./data1/arc-agi_test_challenges.json"
    if not os.path.exists(challenge_path):
        challenge_path = "/root/EdgeTRM/arc-prize-2025/arc-agi_test_challenges.json"
        
    try:
        with open(challenge_path) as f:
            test_puzzles = json.load(f)
    except Exception:
        pass

    def arc_grid_to_np(grid) -> np.ndarray:
        return np.array(grid, dtype=np.uint8)

    def grid_hash(grid: np.ndarray) -> int:
        return hash(grid.tobytes())

    def get_crop(seq: np.ndarray) -> np.ndarray:
        s = seq[seq != 0]
        if len(s) == 0:
            return np.zeros((1, 1), dtype=np.uint8)
        n = int(len(s) ** 0.5)
        return s[:n*n].reshape(n, n)

    # 1000 shuffling augmentations configurations
    aug_fns = [
        lambda g: g,
        lambda g: np.rot90(g, 1),
        lambda g: np.rot90(g, 2),
        lambda g: np.rot90(g, 3),
        lambda g: np.fliplr(g),
        lambda g: np.flipud(g),
        lambda g: np.transpose(g),
        lambda g: np.rot90(np.transpose(g), 2)
    ]
    inv_fns = [
        lambda g: g,
        lambda g: np.rot90(g, -1),
        lambda g: np.rot90(g, -2),
        lambda g: np.rot90(g, -3),
        lambda g: np.fliplr(g),
        lambda g: np.flipud(g),
        lambda g: np.transpose(g),
        lambda g: np.transpose(np.rot90(g, -2))
    ]

    # Pre-map all 1,000 augmentations to their original puzzle name and inverse transformation function
    # Layout of puzzle_identifiers list
    identifiers_json_path = os.path.join(os.path.dirname(test_dir), "identifiers.json")
    try:
        with open(identifiers_json_path) as f:
            all_puzzle_names = json.load(f)
    except Exception:
        all_puzzle_names = []

    def get_aug(identifier: int):
        puzzle_idx = (identifier - 1) // 8000
        aug_idx = (identifier - 1) % 8000
        orig_name = all_puzzle_names[puzzle_idx] if puzzle_idx < len(all_puzzle_names) else ""
        inv_fn = inv_fns[aug_idx % 8]
        return orig_name, inv_fn

    # High-Performance CSR Batch Reconstruction
    precomputed_input_info = {}
    local_hmap = {}
    local_preds = {}

    t0 = time.time()
    
    ds = loader.dataset
    ds._lazy_load_dataset()
    set_name = list(ds._data.keys())[0]
    dataset = ds._data[set_name]
    
    inputs_np = dataset["inputs"]
    labels_np = dataset["labels"]
    puzzle_indices = dataset["puzzle_indices"]
    puzzle_identifiers = dataset["puzzle_identifiers"]
    group_indices = dataset["group_indices"]
    
    # In fast_mode, we select exactly 1 puzzle per group to speed up evaluation by 1000x!
    if fast_mode:
        selected_sample_indices = []
        selected_pids = []
        for g in range(len(group_indices) - 1):
            puzzle_id = group_indices[g]
            start_sample = puzzle_indices[puzzle_id]
            end_sample = puzzle_indices[puzzle_id + 1]
            for s_idx in range(start_sample, end_sample):
                selected_sample_indices.append(s_idx)
                selected_pids.append(puzzle_identifiers[puzzle_id])
        
        selected_sample_indices = np.array(selected_sample_indices, dtype=np.int32)
        inputs_np = inputs_np[selected_sample_indices]
        labels_np = labels_np[selected_sample_indices]
        pids_np = np.array(selected_pids, dtype=np.int32)
    else:
        pids_np = np.zeros(len(inputs_np), dtype=np.int32)
        puzzle_idx = 0
        for i in range(len(inputs_np)):
            while puzzle_idx + 1 < len(puzzle_indices) and i >= puzzle_indices[puzzle_idx + 1]:
                puzzle_idx += 1
            pids_np[i] = puzzle_identifiers[puzzle_idx]
            
    num_samples = len(inputs_np)
    batch_size = loader.batch_size if (hasattr(loader, "batch_size") and loader.batch_size is not None) else 512
    num_batches = (num_samples + batch_size - 1) // batch_size
    
    pbar = tqdm(range(num_batches), desc="Evaluating ARC Batches", leave=False)
    for batch_idx in pbar:
        if max_batches is not None and batch_idx >= max_batches:
            break

        start_idx = batch_idx * batch_size
        end_idx = min(start_idx + batch_size, num_samples)
        
        x_batch = torch.from_numpy(inputs_np[start_idx:end_idx].copy()).to(device, dtype=torch.long)
        y_true  = torch.from_numpy(labels_np[start_idx:end_idx].copy()).to(device, dtype=torch.long)
        pids    = torch.from_numpy(pids_np[start_idx:end_idx]).to(device, dtype=torch.long)

        batch = {
            "inputs":             x_batch.to(torch.int32),
            "labels":             y_true.to(torch.int32),
            "puzzle_identifiers": pids.to(torch.int32),
        }

        carry = inner.initial_carry(batch)
        ic    = carry.inner_carry
        cast  = lambda t: t.to(device)
        
        from models.recursive_reasoning.trm import TinyRecursiveReasoningModel_ACTV1Carry, TinyRecursiveReasoningModel_ACTV1InnerCarry
        carry = TinyRecursiveReasoningModel_ACTV1Carry(
            inner_carry=TinyRecursiveReasoningModel_ACTV1InnerCarry(
                z_H=cast(ic.z_H), z_L=cast(ic.z_L)),
            steps=carry.steps.to(device),
            halted=carry.halted.to(device),
            current_data={k: v.to(device) for k, v in carry.current_data.items()},
        )

        last_outputs = None
        for _ in range(n_sup_max):
            carry, outputs = inner(carry, batch)
            last_outputs = outputs
            if carry.halted.all():
                break

        if last_outputs is None:
            continue

        preds_batch = last_outputs["logits"].argmax(-1).cpu().numpy()
        q_logits    = last_outputs.get("q_halt_logits", torch.zeros(preds_batch.shape[0], device=device))
        q_values    = q_logits.sigmoid().cpu().numpy().flatten()

        inputs_cpu  = inputs_np[start_idx:end_idx]
        pids_cpu    = pids_np[start_idx:end_idx]

        for i in range(preds_batch.shape[0]):
            identifier = pids_cpu[i]
            if identifier == 0:
                continue

            orig_name, _inverse_fn = get_aug(identifier)
            pred_seq = preds_batch[i]
            q_val = float(q_values[i])

            sample_idx = start_idx + i
            if sample_idx in precomputed_input_info and precomputed_input_info[sample_idx][0] == orig_name:
                input_hash = precomputed_input_info[sample_idx][1]
            else:
                inp_seq = inputs_cpu[i]
                input_grid = _inverse_fn(get_crop(inp_seq))
                input_hash = grid_hash(input_grid)

            pred_grid = _inverse_fn(get_crop(pred_seq))
            pred_hash = grid_hash(pred_grid)

            local_hmap[pred_hash] = pred_grid

            local_preds.setdefault(orig_name, {})
            local_preds[orig_name].setdefault(input_hash, [])
            local_preds[orig_name][input_hash].append((pred_hash, q_val))

    n_puzzles = 0
    correct = [0, 0]
    cell_hits = 0
    n_cells = 0

    evaluated_puzzles = [name for name in test_puzzles.keys() if name in local_preds]
    for name in evaluated_puzzles:
        puzzle = test_puzzles[name]
        n_puzzles += 1
        num_correct = [0, 0]
        for pair in puzzle["test"]:
            inp_grid = arc_grid_to_np(pair["input"])
            out_grid = arc_grid_to_np(pair["output"])
            input_hash = grid_hash(inp_grid)
            label_hash = grid_hash(out_grid)

            p_map = {}
            for h, q in local_preds[name].get(input_hash, []):
                p_map.setdefault(h, [0, 0.0])
                p_map[h][0] += 1
                p_map[h][1] += q

            if not len(p_map):
                continue

            p_candidates = []
            for h, stats in p_map.items():
                freq = stats[0]
                avg_q = stats[1] / stats[0]
                p_candidates.append((h, freq, avg_q))
                
            p_candidates.sort(key=lambda x: (x[1], x[2]), reverse=True)
            
            if p_candidates[0][0] == label_hash:
                num_correct[0] = 1
                
            if len(p_candidates) > 1:
                if p_candidates[0][0] == label_hash or p_candidates[1][0] == label_hash:
                    num_correct[1] = 1
            else:
                if p_candidates[0][0] == label_hash:
                    num_correct[1] = 1

        correct[0] += num_correct[0]
        correct[1] += num_correct[1]

        # Cell Accuracy
        for pair in puzzle["test"]:
            inp_grid = arc_grid_to_np(pair["input"])
            out_grid = arc_grid_to_np(pair["output"])
            input_hash = grid_hash(inp_grid)

            preds_list = local_preds.get(name, {}).get(input_hash, [])
            if not preds_list:
                continue

            p_map = {}
            for h, q in preds_list:
                p_map.setdefault(h, [0, 0.0])
                p_map[h][0] += 1
                p_map[h][1] += q
            
            p_candidates = []
            for h, stats in p_map.items():
                freq = stats[0]
                avg_q = stats[1] / stats[0]
                p_candidates.append((h, freq, avg_q))
            p_candidates.sort(key=lambda x: (x[1], x[2]), reverse=True)
            
            top_hash = p_candidates[0][0]
            top_grid = local_hmap[top_hash]

            if top_grid.shape == out_grid.shape:
                cell_hits += (top_grid == out_grid).sum()
                n_cells += out_grid.size
            else:
                n_cells += out_grid.size

    cell_acc = cell_hits / n_cells if n_cells > 0 else 0.0
    pass_1_acc = correct[0] / n_puzzles if n_puzzles > 0 else 0.0
    pass_2_acc = correct[1] / n_puzzles if n_puzzles > 0 else 0.0
    elapsed = time.time() - t0
    ms_per_puzzle = elapsed / n_puzzles if n_puzzles > 0 else 0.0

    if return_pass2:
        return pass_1_acc, pass_2_acc, cell_acc, ms_per_puzzle * 1000, n_puzzles
    else:
        return pass_1_acc, cell_acc, ms_per_puzzle * 1000, n_puzzles

def get_inner(m):
    """Unwrap compiled/DDP model."""
    m2 = m.module if hasattr(m, 'module') else m
    return m2._orig_mod if hasattr(m2, '_orig_mod') else m2


---
## Section 5 — Training a TRM Model from Scratch

In this section, we implement a premium-grade training pipeline to train a fresh `TinyRecursiveReasoningModel_ACTV1` model from scratch on the ARC-AGI dataset.

### Training Details & Hyperparameters
- **Main Optimizer**: `AdamAtan2` (Atan2-based gradient updates for superior learning) with fallback to `AdamW` if not installed.
- **Embedding Optimizer**: `CastedSparseEmbeddingSignSGD_Distributed` to update sparse puzzle embeddings using SignSGD.
- **Learning Rate Schedule**: Cosine learning rate decay with a linear warmup phase.
- **Model Configuration**: Paper-aligned architecture ($H_{cycles}=4, L_{cycles}=4, L_{layers}=2, hidden\_size=512$, vocab_size=12, seq_len=900, puzzle embedding dimension 512, halt steps 16).
- **VRAM Management**: Batch size of `256` prevents GPU memory swapping and PCIe bottlenecking, ensuring maximum local execution speed.

In [5]:
# # ── 5.0  Dataset Generation ──────────────────────────────────────────────────
# # Run this cell to build the Maze-Hard dataset.
# # This downloads from HF hub and generates the required numpy memmaps.

! python3 TinyRecursiveModels/dataset/build_maze_dataset.py --output-dir data/maze-30x30-hard-1k


In [5]:
# ── 5.1  Model & Dataset Initialization ──────────────────────────────────────
import os
import sys
import math
import copy
import time
import torch
from torch import nn
from torch.utils.data import DataLoader
import numpy as np
from tqdm.notebook import tqdm

# Add repo to path if needed
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from puzzle_dataset import PuzzleDataset, PuzzleDatasetConfig
from models.recursive_reasoning.trm import TinyRecursiveReasoningModel_ACTV1, TinyRecursiveReasoningModel_ACTV1Config
from models.losses import ACTLossHead
from models.sparse_embedding import CastedSparseEmbeddingSignSGD_Distributed

# Attempt to load AdamAtan2 from paper, fallback to AdamW
try:
    from adam_atan2_pytorch import AdamAtan2
    print("Successfully imported AdamAtan2!")
except ModuleNotFoundError:
    from torch.optim import AdamW as AdamAtan2
    print("WARNING: adam_atan2_pytorch not found, using AdamW as fallback.")

# Hyperparameters
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_DIR = "./data/maze-30x30-hard-1k"
CHECKPOINT_DIR = "./checkpoints/trm_scratch"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── Optimizing for NVIDIA H200 (141GB VRAM) ───────────────────────────────────
BATCH_SIZE = 512  # Fully saturates Hopper Tensor Cores without swapping
# BATCH_SIZE = 256  # VRAM-friendly batch size to prevent swapping/PCIe bottlenecks
LR = 1e-4          # Scaled learning rate for larger batch size
PUZZLE_EMB_LR = 1e-2
WEIGHT_DECAY = 1.0
PUZZLE_EMB_WEIGHT_DECAY = 1.0
TOTAL_STEPS = 60000
WARMUP_STEPS = 2000
LR_MIN_RATIO = 0.1
CHECKPOINT_INTERVAL = 500
EVAL_INTERVAL = 1000
NUM_PUZZLE_IDENTIFIERS = 927075

# Resume options
RESUME = True  # Set to True to automatically resume from the latest step_*.pt checkpoint

NUM_PUZZLE_IDENTIFIERS = 927075

# Resume options
RESUME = True  # Set to True to automatically resume from the latest step_*.pt checkpoint

print(f"Device: {DEVICE}")
print(f"Initializing loaders from: {DATA_DIR}")

# 1. Build Datasets & Loaders
train_ds_config = PuzzleDatasetConfig(
    seed=0,
    dataset_paths=[DATA_DIR],
    global_batch_size=BATCH_SIZE,
    test_set_mode=False,
    epochs_per_iter=16,  # Set to 16 to pack large batch sizes without StopIteration dropping
    rank=0,
    num_replicas=1
)
train_ds = PuzzleDataset(train_ds_config, split="train") # Load demonstration pairs

test_ds_config = PuzzleDatasetConfig(
    seed=0,
    dataset_paths=[DATA_DIR],
    global_batch_size=BATCH_SIZE,
    test_set_mode=True,
    epochs_per_iter=16,  # Set to 16 to pack large batch sizes without StopIteration dropping
    rank=0,
    num_replicas=1
)
test_ds = PuzzleDataset(test_ds_config, split="test") # Load test challenge pairs

train_loader = DataLoader(train_ds, batch_size=None, num_workers=1, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=None, num_workers=1, pin_memory=True)

print("Loading dataset metadata...")
metadata = train_ds.metadata
print(f"Vocab Size: {metadata.vocab_size}, Sequence Length: {metadata.seq_len}")
print(f"Total groups: {metadata.total_groups}, Total puzzles: {metadata.total_puzzles}")

# 2. Instantiate TRM Model with Loss Head
model_config = {
    "batch_size": BATCH_SIZE,
    "seq_len": metadata.seq_len,
    "puzzle_emb_ndim": 512,
    "num_puzzle_identifiers": metadata.num_puzzle_identifiers,
    "vocab_size": metadata.vocab_size,
    "H_cycles": 3,
    "L_cycles": 4,
    "H_layers": 0,
    "L_layers": 2,
    "hidden_size": 512,
    "expansion": 4.0,
    "num_heads": 8,
    "pos_encodings": "rope",
    "halt_max_steps": 16,
    "halt_exploration_prob": 0.1,
    "forward_dtype": "bfloat16",
    "mlp_t": False,
    "puzzle_emb_len": 16,
    "no_ACT_continue": True
}

print("Initializing fresh TinyRecursiveReasoningModel_ACTV1 model directly on device...")
with torch.device(DEVICE):
    base_model = TinyRecursiveReasoningModel_ACTV1(model_config)
    model = ACTLossHead(base_model, loss_type="stablemax_cross_entropy")

# Force local_weights to be a leaf tensor on DEVICE to satisfy PyTorch Optimizer requirements
emb = model.model.puzzle_emb
if hasattr(emb, 'local_weights') and not emb.local_weights.is_leaf:
    with torch.no_grad():
        leaf_weights = emb.local_weights.to(DEVICE).detach().clone().requires_grad_(True)
        emb.local_weights = nn.Buffer(leaf_weights, persistent=False)
    print("✓ Re-bound local_weights buffer as a leaf tensor on device.")

# Count parameters
num_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {num_params:,}")

# 3. Instantiate Optimizers
# Embedding SignSGD optimizer
emb_optimizer = CastedSparseEmbeddingSignSGD_Distributed(
    model.model.puzzle_emb.buffers(),
    world_size=1,
    lr=PUZZLE_EMB_LR,
    weight_decay=PUZZLE_EMB_WEIGHT_DECAY
)

# Model AdamAtan2 / AdamW optimizer
main_optimizer = AdamAtan2(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.95)
)



Successfully imported AdamAtan2!
Device: cuda
Initializing loaders from: ./data1/arc2train-aug-1000
Loading dataset metadata...
Vocab Size: 12, Sequence Length: 900
Total groups: 1000, Total puzzles: 927072
Initializing fresh TinyRecursiveReasoningModel_ACTV1 model directly on device...
Model parameters: 6,829,058
Model, datasets, and optimizers successfully initialized!


In [ ]:
# ── 5.2  Training & Validation Loop ──────────────────────────────────────────
def get_lr_factor(step, total_steps, warmup_steps, min_ratio):
    if step < warmup_steps:
        return float(step) / float(max(1, warmup_steps))
    
    progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return min_ratio + max(0.0, (1.0 - min_ratio) * 0.5 * (1.0 + math.cos(math.pi * progress)))

start_step = 1
if RESUME:
    import glob
    import re
    checkpoints = glob.glob(os.path.join(CHECKPOINT_DIR, "step_*.pt"))
    if checkpoints:
        # Parse step numbers from filenames
        steps_found = []
        for cp in checkpoints:
            match = re.search(r'step_(\d+)\.pt$', cp)
            if match:
                steps_found.append((int(match.group(1)), cp))
        if steps_found:
            # Sort by step number descending
            steps_found.sort(key=lambda x: x[0], reverse=True)
            latest_step, latest_cp = steps_found[0]
            print(f"[RESUME] Found existing checkpoints. Loading latest: {latest_cp}...")
            # Load model state dict directly on DEVICE
            state_dict = torch.load(latest_cp, map_location=DEVICE)
            # Clean DDP/compile prefixes from state dict
            unwanted_prefix = '_orig_mod.model.'
            clean_state_dict = {}
            for k, v in state_dict.items():
                if k.startswith(unwanted_prefix):
                    clean_state_dict[k[len(unwanted_prefix):]] = v
                elif k.startswith('model.'):
                    clean_state_dict[k[len('model.'):]] = v
                else:
                    clean_state_dict[k] = v

            # Dynamically resize and copy puzzle embeddings to handle dataset transitions
            puzzle_emb_name = 'model.inner.puzzle_emb.weights'
            if puzzle_emb_name not in clean_state_dict:
                puzzle_emb_name = 'inner.puzzle_emb.weights'
            expected_shape = model.model.puzzle_emb.weights.shape
            if puzzle_emb_name in clean_state_dict:
                puzzle_emb = clean_state_dict[puzzle_emb_name]
                if puzzle_emb.shape != expected_shape:
                    print(f'[RESIZE] Resizing puzzle embedding table. Found {puzzle_emb.shape}, Expected {expected_shape}')
                    new_weights = torch.empty(expected_shape, dtype=puzzle_emb.dtype, device=puzzle_emb.device)
                    mean_emb = torch.mean(puzzle_emb, dim=0)
                    new_weights[:] = mean_emb
                    min_rows = min(puzzle_emb.shape[0], expected_shape[0])
                    new_weights[:min_rows] = puzzle_emb[:min_rows]
                    clean_state_dict[puzzle_emb_name] = new_weights

            model.load_state_dict(clean_state_dict, strict=False)
            start_step = latest_step + 1
            print(f"[RESUME] Successfully restored model weights. Resuming training from Step {start_step}!")
        else:
            print("[RESUME] No valid step checkpoints found in directory. Starting from scratch.")
    else:
        print("[RESUME] No checkpoints found. Starting from scratch.")
else:
    print("Starting training from scratch...")

model.train()
model.train()

train_iter = iter(train_loader)
running_loss = 0.0
running_acc = 0.0
running_em = 0.0
running_steps = 0.0
log_window = 100

pbar = tqdm(range(start_step, TOTAL_STEPS + 1), desc="Training steps", initial=start_step-1, total=TOTAL_STEPS)
for step in pbar:
    try:
        set_name, batch, eff_batch_size = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        set_name, batch, eff_batch_size = next(train_iter)
        
    # Scale learning rates according to scheduler
    lr_scale = get_lr_factor(step, TOTAL_STEPS, WARMUP_STEPS, LR_MIN_RATIO)
    for param_group in main_optimizer.param_groups:
        param_group['lr'] = LR * lr_scale
    for param_group in emb_optimizer.param_groups:
        param_group['lr'] = PUZZLE_EMB_LR * lr_scale
        
    # Move batch to device
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    
    # Forward pass
    with torch.device(DEVICE):
        carry = model.initial_carry(batch)
        carry, loss, metrics, _, _ = model(carry=carry, batch=batch, return_keys=[])
    
    # Backward pass & Optimize
    loss_normalized = loss / BATCH_SIZE
    loss_normalized.backward()
    
    # Apply steps & Zero grads
    main_optimizer.step()
    main_optimizer.zero_grad()
    
    emb_optimizer.step()
    emb_optimizer.zero_grad()
    
    # Update metrics
    count = max(float(metrics.get("count", BATCH_SIZE)), 1.0)
    step_loss = float(loss.item()) / BATCH_SIZE
    step_acc = float(metrics.get("accuracy", 0.0)) / count
    step_em = float(metrics.get("exact_accuracy", 0.0)) / count
    step_steps = float(metrics.get("steps", 0.0)) / count
    
    step_idx = step - start_step + 1
    running_loss += (step_loss - running_loss) / min(step_idx, log_window)
    running_acc += (step_acc - running_acc) / min(step_idx, log_window)
    running_em += (step_em - running_em) / min(step_idx, log_window)
    running_steps += (step_steps - running_steps) / min(step_idx, log_window)
    
    pbar.set_postfix({
        "loss": f"{running_loss:.4f}",
        "acc": f"{running_acc*100:.2f}%",
        "em": f"{running_em*100:.2f}%",
        "steps": f"{running_steps:.1f}"
    })
    
    # Periodic evaluation and checkpointing
    if step % CHECKPOINT_INTERVAL == 0:
        # Save checkpoint
        checkpoint_path = os.path.join(CHECKPOINT_DIR, f"step_{step}.pt")
        torch.save(model.state_dict(), checkpoint_path)
        print(f"\n[Step {step}] Checkpoint saved to: {checkpoint_path}")

    if step % EVAL_INTERVAL == 0:

        # Run fast validation evaluation
        print(f"[Step {step}] Running fast per-puzzle evaluation...")
        p1, cell, ms, npuzz = evaluate_arc_per_puzzle(
            model, test_loader, device=DEVICE, n_sup_max=16, max_batches=None, return_pass2=False
        )
        print(f"Validation Results -> Pass@1: {p1*100:.2f}% | Cell Acc: {cell*100:.2f}% | Latency: {ms:.2f} ms/puzzle")
        model.train()  # Restore training mode

[RESUME] No checkpoints found. Starting from scratch.


Training steps:   0%|          | 0/50000 [00:00<?, ?it/s]


[Step 500] Checkpoint saved to: ./checkpoints/trm_scratch/step_500.pt

[Step 1000] Checkpoint saved to: ./checkpoints/trm_scratch/step_1000.pt
[Step 1000] Running fast per-puzzle evaluation...


Evaluating per-puzzle batches:   0%|          | 0/3 [00:00<?, ?it/s]

Validation Results -> Pass@1: 0.00% | Cell Acc: 21.70% | Latency: 42.53 ms/puzzle

[Step 1500] Checkpoint saved to: ./checkpoints/trm_scratch/step_1500.pt

[Step 2000] Checkpoint saved to: ./checkpoints/trm_scratch/step_2000.pt
[Step 2000] Running fast per-puzzle evaluation...


Evaluating per-puzzle batches:   0%|          | 0/3 [00:00<?, ?it/s]

Validation Results -> Pass@1: 0.00% | Cell Acc: 21.88% | Latency: 38.35 ms/puzzle

[Step 2500] Checkpoint saved to: ./checkpoints/trm_scratch/step_2500.pt
